# Exp11.0 v4 — frozen-L1 temporal representation control

Aggregation-only notebook for `d0_d1_l1mem2_context_fusion_frozen_l1_v4`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in (p, *p.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('repo root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_11_0_rsnn_history_internalization' / 'd0_d1_l1mem2_context_fusion_frozen_l1_v4'
runs = pd.read_csv(root / 'run_metrics.csv')
methods = pd.read_csv(root / 'method_summary.csv')
contrasts = pd.read_csv(root / 'paired_contrast_summary.csv')
interactions = pd.read_csv(root / 'interaction_summary.csv')
gaps = pd.read_csv(root / 'temporal_gap_summary.csv')
d0_sources = pd.read_csv(root / 'd0_source_metrics.csv')
runs


## L1 representation preservation
Frozen L1 should have zero relative drift and cosine similarity 1. Trainable pretrained L1 quantifies WCCE-driven representation change.


In [ ]:
drift_cols = ['variant','l1_init','architecture_case','topology','fusion','seed','l1_relative_frobenius_drift','l1_cosine_similarity_to_source','l1_pre_valid_fixed250_ba','l1_pre_valid_whole_ba','l1_comm_valid_fixed250_ba','l1_comm_valid_whole_ba','native_test_ba']
runs[drift_cols].sort_values(['variant','l1_init','architecture_case','topology','fusion','seed'])


## Trainable vs frozen pretrained L1
This is the direct estimate of how much final performance depends on allowing WCCE to reshape L1.


In [ ]:
trainable_minus_frozen = contrasts[contrasts['contrast'] == 'pretrained_trainable_minus_frozen'].copy()
trainable_minus_frozen


## A/B/C/D native performance by L1 mode


In [ ]:
cols = ['variant','l1_init','architecture_case','topology','fusion','seed','native_test_ba','readout_comm_valid_whole_ba','readout_comm_valid_fixed250_ba','readout_comm_valid_temporal_gap']
runs[cols].sort_values(['variant','l1_init','architecture_case','topology','fusion','seed'])


## Frozen-L1 recurrence effect
B-A and D-C under `pretrained_frozen` test whether recurrence can exploit the original L1 representation without changing it.


In [ ]:
frozen_recurrence = contrasts[(contrasts['l1_init'] == 'pretrained_frozen') & contrasts['contrast'].isin(['diagonal_minus_ff','dense_minus_ff'])].copy()
frozen_recurrence


## Recurrence × Fusion interaction


In [ ]:
recurrence_x_fusion = interactions[interactions['interaction'] == 'recurrence_x_fusion'].copy()
recurrence_x_fusion


## Paired D1-D0 effect


In [ ]:
d1_minus_d0 = contrasts[contrasts['contrast'] == 'd1_minus_d0'].copy()
d1_minus_d0


## Temporal internalization
Compare Fixed250-minus-whole from L1 to context/RSNN to final readout, with special attention to pretrained_frozen.


In [ ]:
gap_view = gaps[['variant','l1_init','architecture_case','topology','fusion','layer','support','fixed250_ba_mean','whole_ba_mean','temporal_gap_mean']]
gap_view.sort_values(['variant','l1_init','architecture_case','topology','fusion','support','layer'])


In [ ]:
plot_data = gaps[(gaps['support'] == 'valid') & (gaps['l1_init'] == 'pretrained_frozen')].copy()
for (variant, topology, fusion), group in plot_data.groupby(['variant','topology','fusion']):
    ordered = group.set_index('layer').loc[['l1','rsnn','readout']].reset_index()
    plt.figure()
    plt.plot(ordered['layer'], ordered['temporal_gap_mean'], marker='o')
    plt.axhline(0, linewidth=1)
    plt.ylabel('Fixed250 BA - Whole BA')
    plt.title(f'frozen L1 / {variant} / {topology} / fusion={fusion}')
    plt.show()


## D0 matched-source sanity check


In [ ]:
d0_sources
